# RNN

In [1]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader

In [295]:
df = pd.read_csv("/Users/animeshsingh/Desktop/Datasets/general_knowledge_qa.csv")
df.head()

,question,answer,question_type,image
0,How many days do we have in a week?,Seven,General Knowledge For Kids,NaN
1,How many days are there in a normal year?,365 (not a leap year),General Knowledge For Kids,NaN
2,How many colors are there in a rainbow?,7,General Knowledge For Kids,NaN
3,Which animal is known as the ‘Ship of the Dese...,Camel,General Knowledge For Kids,NaN
4,How many letters are there in the English alph...,26,General Knowledge For Kids,NaN


In [296]:
df = df.drop(columns = ['question_type', 'image'])
df.head()

,question,answer
0,How many days do we have in a week?,Seven
1,How many days are there in a normal year?,365 (not a leap year)
2,How many colors are there in a rainbow?,7
3,Which animal is known as the ‘Ship of the Dese...,Camel
4,How many letters are there in the English alph...,26


In [297]:
df.shape

(930, 2)

In [298]:
unique_q = []
unique_a = []

def tokenize(text):
    text = text.replace('?', ' ')
    text = text.replace("'", ' ')
    text = text.replace('(', ' ')
    text = text.replace(')', ' ')
    text = text.lower()
    return text.split()

for sent in df['question']:
    sent = tokenize(sent)
    unique_q.append(sent)

for sent in df['answer']:
    sent = tokenize(sent)
    unique_a.append(sent)

In [299]:
vocab = {'<UNK>' : 0}

In [300]:
type(unique_q)

list

In [301]:
unique_q = [item for lst in unique_q for item in lst]
unique_a = [item for lst in unique_a for item in lst]

In [302]:
len(unique_q), len(unique_a)

(7508, 1632)

In [303]:
unique_q = set(unique_q)
unique_a = set(unique_a)

In [304]:
len(unique_q), len(unique_a)

(1699, 937)

In [305]:
unique_q = list(unique_q)
unique_a = list(unique_a)

In [306]:
vocab = unique_q + unique_a

In [307]:
len(vocab)

2636

In [308]:
vocab = list(set(vocab))

In [309]:
len(vocab)

2309

In [310]:
type(vocab)

list

In [311]:
vocabulary = {'<PAD>' : 0, '<UNK>' : 1, '<SOS>' : 2, '<EOS>' : 3}

In [312]:
tokens = [i for i in range(len(vocabulary), len(vocab) + 1)]

In [313]:
deect = dict(zip(vocab, tokens))

In [314]:
deect

{'asking': 4,
 'sun': 5,
 'saarc': 6,
 'norway': 7,
 'stars': 8,
 'into': 9,
 'none': 10,
 'founder': 11,
 'archimedes': 12,
 'people.': 13,
 'strings': 14,
 'full': 15,
 '916': 16,
 'will': 17,
 'reappointed': 18,
 'bay': 19,
 '27*0': 20,
 'edison': 21,
 'rbi': 22,
 'cow': 23,
 'lies': 24,
 'was': 25,
 'plant': 26,
 'year.': 27,
 'buddhist': 28,
 'berners-lee': 29,
 'beehives': 30,
 'thousand': 31,
 'nuclear': 32,
 'common': 33,
 'tedros': 34,
 'history': 35,
 'carry': 36,
 'a.': 37,
 'mango': 38,
 'letters': 39,
 'vowels': 40,
 'kalaam,': 41,
 'baku': 42,
 'cross': 43,
 'land': 44,
 'hat-trick': 45,
 'york': 46,
 'mother,': 47,
 'robot': 48,
 'x-rays': 49,
 'ella': 50,
 'frescos': 51,
 '2': 52,
 'treats': 53,
 'beri-beri': 54,
 'eraser,': 55,
 'a': 56,
 'read-only': 57,
 '____________.': 58,
 'environment': 59,
 'ryder': 60,
 'malic': 61,
 'atmosphere': 62,
 'council': 63,
 'thomas': 64,
 'plants': 65,
 'nirvana': 66,
 'sometimes': 67,
 'sky': 68,
 'distance': 69,
 'eggs': 70,
 'am':

In [315]:
vocabulary = vocabulary | deect

In [316]:
vocabulary

{'<PAD>': 0,
 '<UNK>': 1,
 '<SOS>': 2,
 '<EOS>': 3,
 'asking': 4,
 'sun': 5,
 'saarc': 6,
 'norway': 7,
 'stars': 8,
 'into': 9,
 'none': 10,
 'founder': 11,
 'archimedes': 12,
 'people.': 13,
 'strings': 14,
 'full': 15,
 '916': 16,
 'will': 17,
 'reappointed': 18,
 'bay': 19,
 '27*0': 20,
 'edison': 21,
 'rbi': 22,
 'cow': 23,
 'lies': 24,
 'was': 25,
 'plant': 26,
 'year.': 27,
 'buddhist': 28,
 'berners-lee': 29,
 'beehives': 30,
 'thousand': 31,
 'nuclear': 32,
 'common': 33,
 'tedros': 34,
 'history': 35,
 'carry': 36,
 'a.': 37,
 'mango': 38,
 'letters': 39,
 'vowels': 40,
 'kalaam,': 41,
 'baku': 42,
 'cross': 43,
 'land': 44,
 'hat-trick': 45,
 'york': 46,
 'mother,': 47,
 'robot': 48,
 'x-rays': 49,
 'ella': 50,
 'frescos': 51,
 '2': 52,
 'treats': 53,
 'beri-beri': 54,
 'eraser,': 55,
 'a': 56,
 'read-only': 57,
 '____________.': 58,
 'environment': 59,
 'ryder': 60,
 'malic': 61,
 'atmosphere': 62,
 'council': 63,
 'thomas': 64,
 'plants': 65,
 'nirvana': 66,
 'sometimes': 

In [317]:
df['token_q'] = df['question'].apply(lambda text: [vocabulary.get(word, 1) for word in tokenize(text)])
df['token_a'] = df['answer'].apply(lambda text: [vocabulary.get(word, 1) for word in tokenize(text)])

In [318]:
df['token_a'] = [[2] + sent + [3] for sent in df['token_a']]

In [319]:
df['token_q'] = [sent + [0]*(max_q_len - len(sent))for sent in df['token_q']]

In [320]:
df['token_a'] = [sent + [0]*(max_a_len - len(sent))for sent in df['token_a']]

In [321]:
df

,question,answer,token_q,token_a
0,How many days do we have in a week?,Seven,"[747, 1762, 765, 786, 632, 991, 821, 56, 462, ...","[2, 2169, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,How many days are there in a normal year?,365 (not a leap year),"[747, 1762, 765, 1801, 2154, 821, 56, 1659, 11...","[2, 1690, 1592, 56, 1047, 1127, 3, 0, 0, 0, 0,..."
2,How many colors are there in a rainbow?,7,"[747, 1762, 820, 1801, 2154, 821, 56, 1186, 0,...","[2, 1329, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Which animal is known as the ‘Ship of the Dese...,Camel,"[912, 1422, 2186, 886, 2167, 354, 1390, 110, 3...","[2, 1532, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,How many letters are there in the English alph...,26,"[747, 1762, 39, 1801, 2154, 821, 354, 114, 116...","[2, 311, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."
...,...,...,...,...
925,Where are the HeadQuarters of UNESCO?,"Paris, France","[507, 1801, 354, 1053, 110, 1758, 0, 0, 0, 0, ...","[2, 198, 2012, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."
926,Which are the colors of the rings of the Olymp...,"Black, Red, Blue, Green, and Yellow","[912, 1801, 354, 820, 110, 354, 1814, 110, 354...","[2, 1854, 130, 1467, 2163, 87, 2024, 3, 0, 0, ..."
927,Who invented ‘Lift’?,Elisha Graves Otis,"[396, 683, 1543, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[2, 1664, 1891, 1892, 3, 0, 0, 0, 0, 0, 0, 0, ..."
928,Which river crosses the equator twice?,Congo River,"[912, 1207, 563, 354, 548, 1298, 0, 0, 0, 0, 0...","[2, 334, 1207, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."


In [322]:
max_q_len = max(len(q) for q in df['token_q'])
max_a_len = max(len(a) for a in df['token_a'])

In [323]:
print(max_q_len, max_a_len)

28 21


In [324]:
vocabulary

{'<PAD>': 0,
 '<UNK>': 1,
 '<SOS>': 2,
 '<EOS>': 3,
 'asking': 4,
 'sun': 5,
 'saarc': 6,
 'norway': 7,
 'stars': 8,
 'into': 9,
 'none': 10,
 'founder': 11,
 'archimedes': 12,
 'people.': 13,
 'strings': 14,
 'full': 15,
 '916': 16,
 'will': 17,
 'reappointed': 18,
 'bay': 19,
 '27*0': 20,
 'edison': 21,
 'rbi': 22,
 'cow': 23,
 'lies': 24,
 'was': 25,
 'plant': 26,
 'year.': 27,
 'buddhist': 28,
 'berners-lee': 29,
 'beehives': 30,
 'thousand': 31,
 'nuclear': 32,
 'common': 33,
 'tedros': 34,
 'history': 35,
 'carry': 36,
 'a.': 37,
 'mango': 38,
 'letters': 39,
 'vowels': 40,
 'kalaam,': 41,
 'baku': 42,
 'cross': 43,
 'land': 44,
 'hat-trick': 45,
 'york': 46,
 'mother,': 47,
 'robot': 48,
 'x-rays': 49,
 'ella': 50,
 'frescos': 51,
 '2': 52,
 'treats': 53,
 'beri-beri': 54,
 'eraser,': 55,
 'a': 56,
 'read-only': 57,
 '____________.': 58,
 'environment': 59,
 'ryder': 60,
 'malic': 61,
 'atmosphere': 62,
 'council': 63,
 'thomas': 64,
 'plants': 65,
 'nirvana': 66,
 'sometimes': 

In [325]:
def text2tok(text, vocabulary):
    return [vocabulary.get(word, vocabulary['<UNK>']) for word in tokenize(text)]

In [326]:
text2tok("what is apple?", vocabulary)

[607, 2186, 1894]

In [327]:
class datalao(Dataset):
    def __init__(self, questions, answers):
        self.q = questions
        self.a = answers

    def __len__(self):
        return self.q.shape[0]

    def __getitem__(self, idx):
        return torch.tensor(self.q[idx]), torch.tensor(self.a[idx])

In [328]:
dataset = datalao(df['token_q'], df['token_a'])

In [329]:
dataset[0]

(tensor([ 747, 1762,  765,  786,  632,  991,  821,   56,  462,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0]),
 tensor([   2, 2169,    3,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0]))

In [330]:
train_load = DataLoader(dataset, batch_size = 4, shuffle = True)

In [331]:
for q, a in train_load:
    print(q,a)

tensor([[  56, 1653, 1009,  856, 1340, 2186,  573,   56, 1524,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0],
        [ 912, 2214, 2186,  886, 2167, 1180, 2214,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0],
        [ 912, 2186,  354, 1443,  810,  821,  354,  410,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0],
        [ 747, 1762, 1818, 1439, 1563, 1973,  196,  991,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0]]) tensor([[   2,  537,    3,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0],
        [   2,  426,    3,    0,    0,    0,    0,    0,    0,    0,  

In [332]:
print(q.shape,a.shape)

torch.Size([2, 28]) torch.Size([2, 21])


In [333]:
class encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(
        vocab_size, 
        embed_dim,
        padding_idx = 0
        )
        self.gru = nn.GRU(
        embed_dim,
        hidden_dim,
        batch_first = True
        )

    def forward(self, x):
        embed = self.embedding(x)
        outputs, hidden = self.gru(embed)
        return hidden

In [334]:
class decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(
        vocab_size,
        embed_dim,
        padding_idx = 0
        )
        self.gru = nn.GRU(
        embed_dim,
        hidden_dim,
        batch_first = True
        )
        self.classif = nn.Linear(
        hidden_dim,
        vocab_size
        )

    def forward(self, tok, hid):
        embed = self.embedding(tok)
        output, hid = self.gru(embed, hid)
        logits = self.classif(output)
        return logits, hid

In [335]:
class seq2seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, q, a):
        hid = self.encoder(q)
        decoder_input = a[:, :-1]
        logits, _ = self.decoder(decoder_input, hid)

        return logits

In [336]:
encoder = encoder(
    vocab_size = len(vocabulary),
    embed_dim = 64,
    hidden_dim = 128,
)

decoder = decoder(
    vocab_size = len(vocabulary),
    embed_dim = 64,
    hidden_dim = 128,
)

model = seq2seq(
    encoder,
    decoder
)

In [346]:
crit = nn.CrossEntropyLoss(ignore_index = 0)
lr = 0.01
epochs = 100

In [347]:
optimizer = torch.optim.Adam(model.parameters(), lr = lr)

In [348]:
for epoch in range(epochs):
    tot_loss = 0
    for q, a in train_load:
        logits = model(q, a)
        targets = a[:, 1:]
        loss = crit(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        tot_loss += loss.item()

    avg_loss = (tot_loss / len(train_load))
    print(f'epoch: {epoch} loss: {avg_loss}')

epoch: 0 loss: 3.58513653433438
epoch: 1 loss: 3.3348935462495457
epoch: 2 loss: 3.389253042302546
epoch: 3 loss: 3.526460331319052
epoch: 4 loss: 3.584263842568889
epoch: 5 loss: 3.4288249539437228
epoch: 6 loss: 3.542771111806063
epoch: 7 loss: 3.656615276753007
epoch: 8 loss: 3.7056174440381353
epoch: 9 loss: 3.5708072502725625
epoch: 10 loss: 3.7108250235333786
epoch: 11 loss: 3.313891154374176
epoch: 12 loss: 3.0970142626772184
epoch: 13 loss: 3.297169406620522
epoch: 14 loss: 3.270278858663934
epoch: 15 loss: 3.1556495500992296
epoch: 16 loss: 3.1198863980156477
epoch: 17 loss: 2.810615798224627
epoch: 18 loss: 2.851580840252543
epoch: 19 loss: 2.513874232864943
epoch: 20 loss: 2.6921675219709558
epoch: 21 loss: 2.679347505524171
epoch: 22 loss: 2.5324637784418758
epoch: 23 loss: 2.6009889723405304
epoch: 24 loss: 2.7574163412899098
epoch: 25 loss: 2.951385712730718
epoch: 26 loss: 2.49155044536528
epoch: 27 loss: 2.2284132879978853
epoch: 28 loss: 2.2572194597095026
epoch: 29 lo

In [349]:
torch.save(model.state_dict(), 'rnn_qa.pth')

In [351]:
idx_to_word = {v:k for k,v in vocabulary.items()}

In [378]:
def generate(model, question_tensor, SOS, EOS, idx_to_word, max_len = 20):

    model.eval()
    with torch.no_grad():

        if question_tensor.dim() == 1:
            question_tensor = question_tensor.unsqueeze(0)

        hidden = model.encoder(question_tensor)
        token = torch.tensor([[SOS]], dtype = torch.long, device = question_tensor.device)

        generated_tokens = []
        for _ in range(max_len):
            logits, hidden = model.decoder(token, hidden)
            token = logits.argmax(dim = -1)
            predicted_id = token.item()

            if predicted_id == EOS:
                break

            generated_tokens.append(predicted_id)

        words = [idx_to_word[i] for i in generated_tokens]
        return " ".join(words)

In [379]:
q, a = dataset[0]
generate(model, q, 2, 3, idx_to_word)

'seven'

In [380]:
for i in q:
    print (idx_to_word[int(i)])

how
many
days
do
we
have
in
a
week
<PAD>
<PAD>
<PAD>
<PAD>
<PAD>
<PAD>
<PAD>
<PAD>
<PAD>
<PAD>
<PAD>
<PAD>
<PAD>
<PAD>
<PAD>
<PAD>
<PAD>
<PAD>
<PAD>
